# Delphi

In [ ]:
import os
import pickle
import torch
from model import DelphiConfig, Delphi
from tqdm import tqdm
import pandas as pd
import numpy as np
import textwrap

import matplotlib.pyplot as plt
%config InlineBackend.figure_format='retina'

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams.update({'axes.grid': True,
                     'grid.linestyle': ':',
                     'axes.spines.bottom': False,
          'axes.spines.left': False,
          'axes.spines.right': False,
          'axes.spines.top': False})
plt.rcParams['figure.dpi'] = 72
plt.rcParams['pdf.fonttype'] = 42

#Green
light_male = '#BAEBE3'
normal_male = '#0FB8A1'
dark_male = '#00574A'


#Purple
light_female = '#DEC7FF'
normal_female = '#8520F1'
dark_female = '#7A00BF'


delphi_labels = pd.read_csv('delphi_labels_chapters_colours_icd.csv')
labels = pd.read_csv("data/ukb_simulated_data/labels.csv", header=None, sep="\t")

dataset_subset_size = 128

In [ ]:
[np.where(labels[0].str.startswith(x))[0][0] for x in ['A41','B01','C25','C50','G30','E10','F32','I21','J45','Death',]]

## Load model

In [ ]:
out_dir = 'Delphi-2M'
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1', etc.
dtype ='float32' #'bfloat16' # 'float32' or 'bfloat16' or 'float16'
seed = 1337

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

device_type = 'cuda' if 'cuda' in device else 'cpu'
dtype = {'float32': torch.float32, 'float64': torch.float64, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

In [ ]:
ckpt_path = os.path.join(out_dir, 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)
conf = DelphiConfig(**checkpoint['model_args'])
model = Delphi(conf)
state_dict = checkpoint['model']
model.load_state_dict(state_dict)

model.eval()
model = model.to(device)

In [ ]:
checkpoint['model_args']

## Load data

In [ ]:
from utils import get_batch, get_p2i

In [ ]:
train = np.fromfile('data/ukb_simulated_data/train.bin', dtype=np.uint32).reshape(-1,3)
val = np.fromfile('data/ukb_simulated_data/val.bin', dtype=np.uint32).reshape(-1,3)

train_p2i = get_p2i(train)
val_p2i = get_p2i(val)

In [ ]:
person = [('Male', 0),
 ('B01 (varicella [chickenpox])',2),
 ('L20 (atopic dermatitis)',3),
 ('Healthy', 5),
 ('Healthy', 10),
 ('Healthy', 15),
 ('Healthy', 20),
 ('G43 (migraine)', 20),
 ('E73 (lactose intolerance)',21),
 ('B27 (infectious mononucleosis)',22),
 ('Healthy', 25),
 ('J11 (influenza, virus not identified)',28),
 ('Healthy', 30),
 ('Healthy', 35),
 ('Healthy', 40),
 ('Smoking_low', 41),
 ('BMI_mid', 41),
 ('Alcohol_low', 41),
 ('Healthy', 42),
]
person = [(a, b * 365.25) for a,b in person] 

## Generated trajectories

In [ ]:
max_new_tokens = 100
num_samples = 1

x = (torch.tensor([labels[0].to_list().index(x[0]) for x in person], device=device)[None, ...])
a = (torch.tensor([x[1]+0. for x in person], device=device)[None, ...])

res = []
with torch.no_grad():
    for k in range(num_samples):
        y,b,_ = model.generate(x, a, max_new_tokens, top_k=None)
        print("\n".join([f"{['',''][i >= len(person)]}{age:2.1f}: {delphi_labels.loc[d,'name']}" for i, (d, age) in  enumerate(zip(y.cpu().numpy().flatten(), b.cpu().numpy().flatten()/365.))]))
        print(10*"-")

## Calibration of predicted times

In [ ]:
d = get_batch(range(128), val, val_p2i,  select='smart_random', block_size=model.config.block_size, device=device)
with torch.no_grad():
    p = model(*d)[0].cpu().detach().numpy().squeeze()
t = (d[3]-d[1])[:,:].cpu().numpy().squeeze()

In [ ]:
from scipy.special import logsumexp
logsumexp(p,-1).shape

plt.figure(figsize=(4, 4))
expected_t = 1/np.exp(logsumexp(p,-1))
delta_log_t = 0.1
observed_t = [t[(expected_t > 10**i) * (expected_t <= 10**(i+delta_log_t)) * (t>0)  ].mean() for i in np.arange(1.75,4,delta_log_t)]
plt.axes().set_aspect('equal')
plt.scatter(expected_t, t+0.5, marker=".", c='lightgrey', rasterized=True)
plt.xlabel('Expected days to next token')
plt.ylabel('Observed days to next token')
plt.plot(10**(np.arange(1.75,4,delta_log_t)+delta_log_t/2.),observed_t, label='average')
plt.yscale('log')
plt.xscale('log')
plt.legend()
plt.xlim(1,2e3)
plt.ylim(1,2e3)
plt.plot([0,1],[0,1], transform = plt.gca().transAxes, c='k' , ls=(0, (5, 5)), linewidth=0.7)

plt.gca().tick_params(length=1.15, width=0.3, labelsize=8, grid_alpha=1, grid_linewidth=0.45, grid_linestyle=':')
plt.gca().tick_params(length=1.15, width=0.3, labelsize=8, grid_alpha=0.0, grid_linewidth=0.35, which='minor')

## Incidence

In [ ]:
## Load large chunk of data
d = get_batch(range(dataset_subset_size), val, val_p2i,  
              select='smart_random', block_size=64, 
              device=device, padding='random')

In [ ]:
has_gender = np.array([2 in x or 3 in x for x in d[0]])
is_male = np.array([3 in x for x in d[0]])
is_female = np.array([2 in x for x in d[0]])

In [ ]:
w = np.zeros((d[0].shape[0], model.config.vocab_size))
for i,row in enumerate(d[0]):
    for j in row:
        w[i, int(j)]=1

In [ ]:
device = 'cuda'
p = []
model.to(device)
batch_size=512
with torch.no_grad():
    for d_batch in tqdm(zip(*map(lambda x: torch.split(x.to(device), batch_size), d)), total=d[0].shape[0]//batch_size+1):
        p.append(model(*d_batch)[0].cpu().detach())
p = torch.vstack(p)

d = [d_.cpu() for d_ in d]

###  Age-sex incidence baseline

In [ ]:
females = train[np.isin(train[:,0], train[train[:,2]==1,0])]
males = train[np.isin(train[:,0], train[train[:,2]==2,0])]
n_females = (train[:,2]==1).sum()
n_males = (train[:,2]==2).sum()

In [ ]:
n_males = - np.cumsum(np.histogram(np.maximum(40,np.round(males[np.where(males[:-1,0]!=males[1:,0])[0],1]/365.25))+1, np.arange(100))[0]) #males[:,2]==males[:,2].max(),1])
n_males = n_males - n_males[-1]
n_females = - np.cumsum(np.histogram(np.maximum(40,np.round(females[np.where(females[:-1,0]!=females[1:,0])[0],1]/365.25)+1), np.arange(100))[0]) #males[:,2]==males[:,2].max(),1])
n_females = n_females - n_females[-1]

In [ ]:
males_in_ukb = np.cumsum(np.histogram((males[(males[:,2] > 2) * (males[:,2] <=4),1]/365.25).astype('int'), np.arange(100))[0])
females_in_ukb = np.cumsum(np.histogram((females[(females[:,2] > 2) * (females[:,2] <=4),1]/365.25).astype('int'), np.arange(100))[0])

### Modelled age-incidence

In [ ]:
from scipy.interpolate import make_smoothing_spline#splrep, BSpline

In [ ]:
x = d[1][:,:].cpu().detach().numpy()/365.25 #+ np.random.rand(d[1].shape[0])/10
y = p[:,:,k]

In [ ]:
x = d[1][:,:].cpu().detach().numpy()/365.25
j = np.where(np.isin(d[2].cpu(), k).any(axis=1))[0][1]

In [ ]:
w[:,:14]=0
cancer_idx = np.where(labels[0].str.startswith('C00'))[0][0]
cancers = (w.sum(0)[cancer_idx:].argsort()[::-1]+cancer_idx)[:20]

In [ ]:
other_diseases = np.array([x for x in np.where(labels[0].str.contains("infarction|stroke|diabetes") > 0)[0] if w[:,x].sum()>100])
diseases = np.concatenate([[labels[0].to_list().index("Death")], w.sum(0).argsort()[::-1][:35], other_diseases]).astype(int)

In [ ]:
fig, ax = plt.subplots(8,5, figsize=(25,32), sharex=True, sharey=True)
axf = ax.ravel()

for i,k in enumerate(diseases[:40]):
    x = d[1][:,:].detach().numpy()/365.25 #+ np.random.rand(d[1].shape[0])/10
    y = np.exp(p.detach().numpy()[:,:,k]) * 365.25  #* (d[3]-d[1]).clamp(max=365.25, min=1.).numpy()
    y = 1-np.exp(-y) #y/(1+y)
    no_prior_disease = ~np.isin(d[0], k).any(axis=1)
    sub_sample = np.random.randint(0, len(x[has_gender * no_prior_disease].ravel()), 5000)
    axf[i].scatter(x[has_gender * no_prior_disease].ravel()[sub_sample], y[has_gender * no_prior_disease].ravel()[sub_sample], 
                   marker='.', c=np.repeat(np.array(['#DEC7FF','#BAEBE3'])[0+is_male[has_gender * no_prior_disease]],x.shape[1]).ravel()[sub_sample],#'lightgrey',##np.minimum((d[0]>1).cumsum(1).detach().numpy(),19)[has_gender,-1], 
                   edgecolors='white',  s=50, label='all other tokens')
    has_k = np.where(d[2].detach().numpy()[has_gender] == k)[0]
    before_k = d[2].detach().numpy()[has_gender].ravel() == k
    axf[i].scatter(x[has_gender].ravel()[before_k], y[has_gender].ravel()[before_k], 
                   marker='.', c=np.array(['#7A00BF','#00574A'])[0+is_male[has_gender][has_k]],##np.minimum((d[0]>1).cumsum(1).detach().numpy(),19)[has_gender,-1], 
                   edgecolors='white',  s=50, label='last token prior to disease')
    j = np.where(np.isin(d[2], k).any(axis=1))[0][0]
    j0 = np.where(x[j]>=0)[0][0]
    jk = np.where(d[2][j,:].detach().numpy() == k)[0][0]

    axf[i].plot(x[j][j0:jk+1],  y[j][j0:jk+1], ds='steps-post',c='k', ls="-", marker='.',markersize=8,markeredgecolor='white', markerfacecolor='k' , label='selected case')

    axf[i].scatter(x[j][jk],  y[j][jk], marker='.', s=200,edgecolors='white', c='k', zorder=3)

    h,x = np.histogram(females[females[:,2]==k-1,1]/365, np.arange(100))
    axf[i].stairs(h/n_females,x, color='#8520F1', lw=2, label='XX incidence')
    h,x = np.histogram(males[males[:,2]==k-1,1]/365, np.arange(100))
    axf[i].stairs(h/n_males,x, color='#0FB8A1', lw=2, label='XY incidence')
    axf[i].set_ylim((1e-5, 1))
    axf[i].set_xlim((0, None))

    axf[i].set_yscale('log')
    axf[i].set_title("\n".join(textwrap.wrap(labels.iloc[k,0], width=30)), verticalalignment='top', fontsize=10, fontweight='bold')
    if i % ax.shape[1] ==0:
        axf[i].set_ylabel('Rate per year')
        axf[i].legend(loc='upper left')

    if i // ax.shape[1] == ax.shape[0] -1:
        axf[i].set_xlabel('Age')
    #plt.show()
#plt.tight_layout()

In [ ]:
def plot_age_incidence(ix,d,p, highlight_idx=0):
    fig, ax = plt.subplots((len(ix)-1)//5+1,5, figsize=(18,(3*(len(ix)-1)//5+1)), sharex=False, sharey=True)
    axf = ax.ravel()
    for i,k in enumerate(ix):
        x = d[1][:,:].detach().numpy()/365.25 #+ np.random.rand(d[1].shape[0])/10
        y = np.exp(p.detach().numpy()[:,:,k])*365.25
        y = 1-np.exp(-y)
        no_prior_disease = ~np.isin(d[0], k).any(axis=1)
        sub_sample = np.random.randint(0, len(x[has_gender * no_prior_disease].ravel()), 5000)
        axf[i].scatter(x[has_gender * no_prior_disease].ravel()[sub_sample], 
                       y[has_gender * no_prior_disease].ravel()[sub_sample], 
                       marker='.', c=np.repeat(np.array(['#DEC7FF','#BAEBE3'])[0+is_male[has_gender * no_prior_disease]],x.shape[1]).ravel()[sub_sample],#'lightgrey',##np.minimum((d[0]>1).cumsum(1).detach().numpy(),19)[has_gender,-1], 
                       edgecolors='white',  s=50, label='Delphi, all time steps', rasterized=True)
        has_k = np.where(d[2].detach().numpy()[has_gender] == k)[0]
        before_k = d[2].detach().numpy()[has_gender].ravel() == k
        axf[i].scatter(x[has_gender].ravel()[before_k], 
                       y[has_gender].ravel()[before_k], 
                       marker='.', c=np.array(['#7A00BF','#00574A'])[0+is_male[has_gender][has_k]],##np.minimum((d[0]>1).cumsum(1).detach().numpy(),19)[has_gender,-1], 
                       edgecolors='white',  s=50, label='Delphi, penultimate step', rasterized=True)

        j = np.where(np.isin(d[2], k).any(axis=1))[0][highlight_idx]
        j0 = np.where(x[j]>=0)[0][0]
        jk = np.where(d[2][j,:].detach().numpy() == k)[0][0]

        axf[i].plot(x[j][j0:jk+1],  y[j][j0:jk+1], ds='steps-post',c='k', ls="-", marker='.',markersize=8,markeredgecolor='white', markerfacecolor='k' , label='selected case')
        axf[i].scatter(x[j][jk],  y[j][jk], marker='.', s=200,edgecolors='white', c='k', zorder=3)

        h,x = np.histogram(females[females[:,2]==k-1,1]/365.25, np.arange(100))
        axf[i].stairs(h/n_females,x, color='#8520F1', lw=2, label='reported incidence, female')
        h,x = np.histogram(males[males[:,2]==k-1,1]/365.25, np.arange(100))
        axf[i].stairs(h/n_males,x, color='#0FB8A1', lw=2, label='reported incidence, male')
        axf[i].set_ylim((1e-5, 1))
        axf[i].set_xlim((0,80))

        axf[i].set_yscale('log')
        axf[i].set_title("\n".join(textwrap.wrap(delphi_labels.loc[k,'name'], width=30)), verticalalignment='top', fontsize=10, fontweight='bold')
        if i % ax.shape[1] ==0:
            axf[i].set_ylabel('Rate per year')

        if i // ax.shape[1] == ax.shape[0] -1:
            axf[i].set_xlabel('Age')
        
        if i == len(ix)-1:
            axf[i].legend(loc='center left', bbox_to_anchor=(1.05, 0.5))

### Interesting diseases

In [ ]:
diseases_of_interest = [np.where(labels[0].str.startswith(x))[0][0] for x in ['A41','B01','C25','C50','G30','E10','F32','I21','J45','Death',]] #M19
diseases_of_interest

In [ ]:
plot_age_incidence(diseases_of_interest ,d,p, highlight_idx=0)
plt.gcf().tight_layout(h_pad=0.5)
plt.show()

In [ ]:
def get_calibration(j,k,d,p, offset = 365.25, age_groups=range(45,85,5), extrapolate=False, n_samples=3, calibration = 'bins', lifestyle=range(2,12), binning='power', bins=10**np.arange(-6.,1.5,.5)):
    
    l = len(age_groups)
    age_step = age_groups[1]-age_groups[0]
   
    ## Indexes of cases
    wk = np.where(d[2].detach().numpy()==k) 
    
    if len(wk[0])<2:
        return np.repeat(np.nan, l)
  
    ## Select age matched controls    
    wc = (np.array([i for i in range(d[0].shape[0]) if i not in wk[0]]),
          torch.clamp((d[3][np.array([i for i in range(d[0].shape[0]) if i not in wk[0]])] <= d[3][wk][torch.randint(len(wk[0]), size=(d[0].shape[0]-len(wk[0]),1))]).sum(1) + 1, 
                             min=0, max=d[1].shape[1]-1))
    
    ## Indeces of lifestyle
    wl = np.argmax((d[2].detach().numpy() >= 10)*(d[2].detach().numpy() <= 12),1)
    k_after_l = np.array([wl[x[0]] < x[1] for x in zip(list(wk[0]), list(wk[1]))])
    c_after_l = np.array([wl[x[0]] < x[1] for x in zip(list(wc[0]), list(wc[1]))])
    
    r = k_after_l.mean()
    if c_after_l.mean() < r: ## Ie more controls wo lifestyle
        c_sub = np.concatenate([np.where(c_after_l)[0], 
                                np.where(~c_after_l)[0][:int(c_after_l.sum() * (1/r-1))]]) # subsample controls wo lifestyle
    else:
        c_sub = np.concatenate([np.where(c_after_l)[0][:int((~c_after_l).sum() * r/(1-r))],  # subsample controls w lifestyle
                                np.where(~c_after_l)[0]])
    
    wall = (np.concatenate([wk[0], wc[0][c_sub]]), 
            np.concatenate([wk[1], wc[1][c_sub]]))  #-np.ones(d[0].shape[0] - len(wk[0]), dtype=np.int32)]))
        
    
    pred_idx = (d[1][wall[0]] <= d[3][wall].reshape(-1,1) - offset).sum(1) -1
    z = d[1].detach().numpy()[(wall[0], pred_idx)]
    z = z[pred_idx != -1]

    zk = d[3].detach().numpy()[wall] #Target times, cases and controls
    zk = zk[pred_idx != -1] 
    
    if extrapolate:
        x = []
        for i in tqdm(range(len(wall[0]))):
            if pred_idx[i] != -1:
                r_ = []
                #for _ in range(n_samples):
                x_,y_,l_ = model.generate(d[0][[wall[0][i]],:pred_idx[i]+1] * torch.ones([n_samples,1], dtype=torch.uint8), 
                                          d[1][[wall[0][i]],:pred_idx[i]+1] * torch.ones([n_samples,1]), 
                                          max_age=d[1][[wall[0][i]],pred_idx[i]]+offset)
                np.argmax(x_.detach().numpy() != 0,1)
                r_.append(l_.detach().numpy()[0,-1,k])
                x.append(np.log(np.exp(np.array(r_)).mean()))
        x = np.exp(np.array(x))*365.25
 
    else:
        x = np.exp(p[...,j][(wall[0], pred_idx)]) * 365.25
        x = x[pred_idx != -1]
    
    x = x / (1+x) 
    
    wk = (wk[0][pred_idx[:len(wk[0])] != -1], wk[1][pred_idx[:len(wk[0])] != -1])
    
    return x, zk, (wall[0][pred_idx != -1], wall[1][pred_idx != -1])

### Perplexity

In [ ]:
def perplexity(p):
    probs = torch.softmax(p, -1)
    H = -torch.einsum("ijk,ijk->ij",probs, torch.log(probs))
    return torch.exp(H)

In [ ]:
pp = perplexity(p[...,1:])

In [ ]:
incidence_k_g = []
for k in range(len(labels)):
    h_f,x = np.histogram(females[females[:,2]==k-1,1]/365, np.arange(100))
    h_m,x = np.histogram(males[males[:,2]==k-1,1]/365, np.arange(100))
    incidence_k_g.append([h_f/n_females,h_m/n_males])
incidence_k_g = np.array(incidence_k_g)
incidence_k_g[1] = 0.2

In [ ]:
pp_incidence = perplexity(torch.tensor(np.log(np.nan_to_num(incidence_k_g.T)+1e-8))).detach().numpy()

In [ ]:
a =d[1].cpu().detach().numpy()/365.25
plt.scatter(a[has_gender], pp.detach().numpy()[has_gender], marker='.', 
            c=np.repeat(np.array(['#DEC7FF','#BAEBE3'])[0+is_male[has_gender]].ravel(), a.shape[1]),
            edgecolors='white',  s=50)
plt.plot(np.arange(80), pp_incidence[1:81,0], c='#8520F1', ds='steps', label='incidence, female')
plt.plot(np.arange(80), pp_incidence[1:81,1], c='#0FB8A1', ds='steps', label='incidence, males')
for i in range(2):
    ii = [is_female,is_male][i]
    plt.plot(np.arange(80), [np.exp(torch.log(pp[ii][a[ii].astype('int')==i]).detach().numpy().mean()) for i in range(80)], 
             c=['#7A00BF','#00574A'][i], ds='steps', label=['model, females','model, males'][i])
plt.xlim((0,None))
plt.legend(loc='lower right')
plt.yscale('log')
plt.ylabel('Perplexity')
plt.xlabel('Age [yr]')

In [ ]:
for i in range(2):
    ii = [is_female,is_male][i]
    plt.plot(np.arange(80), np.array([np.exp(torch.log(pp[ii][a[ii].astype('int')==i]).detach().numpy().mean()) for i in range(80)])/pp_incidence[1:81,i], 
             c=['#7A00BF','#00574A'][i], ls='', marker='o',label=['females','males'][i], markeredgecolor='white')
plt.ylim((1/2, 2))
#plt.xlim((0.5,None))
plt.xlabel('Age [yr]')
plt.ylabel('Perplexity model/incidence')
plt.yscale('log')
plt.legend()

### Calibration
Load full 100k validation data set

In [ ]:
def auc(x1, x2):
    n1 = len(x1)
    n2 = len(x2)
    R1 = np.concatenate([x1,x2]).argsort().argsort()[:n1].sum() + n1
    U1 = n1*n2 + 0.5*n1*(n1+1) - R1
    if n1 == 0 or n2 == 0:
        return np.nan
    return U1 / n1 / n2

In [ ]:
d100k = get_batch(range(val_p2i.shape[0]-1), val, val_p2i,  
              select='smart_random', block_size=64, 
              device=device, padding='random')

In [ ]:
p100k = []
model.to(device)
batch_size=512
with torch.no_grad():
    for dd in tqdm(zip(*map(lambda x: torch.split(x, batch_size), d100k)), total=d100k[0].shape[0]//batch_size+1):
        p100k.append(model(*[x.to(device) for x in dd])[0].cpu().detach()[:,:,diseases_of_interest].numpy())
p100k = np.vstack(p100k)

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
import scipy
import scipy.stats
from tqdm import tqdm
from scipy.special import logsumexp

def plot_calibration(j,k,d,p, offset = 365.25, age_groups=range(45,85,5), extrapolate=False, n_samples=3, calibration = 'bins', lifestyle_ix=range(10,12), binning='power', bins=10**np.arange(-6.,1.5,.5)):
    
    l = len(age_groups)
    age_step = age_groups[1]-age_groups[0]


    fig, ax = plt.subplots(2,l, figsize=(20/8*l,3), sharex=True, sharey=False, height_ratios=[1, .5])
    ax = ax[np.newaxis,:]
    axf = ax.ravel()
    
    ## Indexes of cases
    wk = np.where(d[2].detach().numpy()==k) 
    
    if len(wk[0])<2:
        return np.repeat(np.nan, l)
  
    wc = np.where(d[2].detach().numpy()!=k) 
    
    if lifestyle_ix is not None:
        ## Indeces of lifestyle
        wl = np.argmax((d[2].detach().numpy() >= min(lifestyle_ix))*(d[2].detach().numpy() <= max(lifestyle_ix)),1)
        k_after_l = np.array([wl[x[0]] < x[1] for x in zip(list(wk[0]), list(wk[1]))])
        c_after_l = np.array([wl[x[0]] < x[1] for x in zip(list(wc[0]), list(wc[1]))])

        r = k_after_l.mean()
        if c_after_l.mean() < r: ## Ie more controls wo lifestyle
            c_sub = np.concatenate([np.where(c_after_l)[0], 
                                    np.where(~c_after_l)[0][:int(c_after_l.sum() * (1/r-1))]]) # subsample controls wo lifestyle
        else:
            c_sub = np.concatenate([np.where(c_after_l)[0][:int((~c_after_l).sum() * r/(1-r))],  # subsample controls w lifestyle
                                    np.where(~c_after_l)[0]])
    else:
        c_sub = range(wc[0].shape[0])
    
    wall = (np.concatenate([wk[0], wc[0][c_sub]]), 
            np.concatenate([wk[1], wc[1][c_sub]]))  #-np.ones(d[0].shape[0] - len(wk[0]), dtype=np.int32)]))
        
    
    pred_idx = (d[1][wall[0]] <= d[3][wall].reshape(-1,1) - offset).sum(1) -1
    z = d[1].detach().numpy()[(wall[0], pred_idx)]
    z = z[pred_idx != -1]

    zk = d[3].detach().numpy()[wall] #Target times, cases and controls
    zk = zk[pred_idx != -1] 
    
    if extrapolate:
        x = []
        for i in tqdm(range(len(wall[0]))):
            if pred_idx[i] != -1:
                r_ = []
                #for _ in range(n_samples):
                x_,y_,l_ = model.generate(d[0][[wall[0][i]],:pred_idx[i]+1] * torch.ones([n_samples,1], dtype=torch.uint8), 
                                          d[1][[wall[0][i]],:pred_idx[i]+1] * torch.ones([n_samples,1]), 
                                          max_age=d[1][[wall[0][i]],pred_idx[i]]+offset)
                np.argmax(x_.detach().numpy() != 0,1)
                r_.append(l_.detach().numpy()[0,-1,k])
                x.append(np.log(np.exp(np.array(r_)).mean()))
        x = np.exp(np.array(x))*365.25
 
    else:
        x = np.exp(p[...,j][(wall[0], pred_idx)]) * 365.25
        x = x[pred_idx != -1]
    
    x = 1 - np.exp(-x * age_step) #x * 1/age_step/ (1/age_step+x) # Rate can't exceed 1/age bin
    
    wk = (wk[0][pred_idx[:len(wk[0])] != -1], wk[1][pred_idx[:len(wk[0])] != -1])
    p_idx = wall[0][pred_idx!=-1]
    
    out = []
    
        
    for i,aa in enumerate(age_groups):
        a = np.logical_and(z / 365.25 >= aa, z / 365.25 < aa+ age_step)
        a *= zk - z < 365.25 #* age_step
        a *= np.isin(np.arange(a.shape[0]),np.unique(p_idx * a, return_index=True)[1]) # Mask duplicated people in age bracket
        axf[i+l].boxplot((x[len(wk[0]):][a[len(wk[0]):]], x[:len(wk[0])][a[:len(wk[0])]]), vert=False, sym='.', widths=.5, whis=(5,95), flierprops = dict(marker='.', markeredgecolor='white', markerfacecolor='k'))
        axf[i+l].set_xscale('log')
        axf[i+l].set_xlim((1e-5, 1))
        axf[i+l].set_yticks((1,2), ['',''])
        if i==0:
            axf[i].set_title(f'{labels.loc[k,0]}\n', fontsize=10, weight='bold', loc='left')
            axf[i].set_ylabel(f'Observed rate [1/yr]')
            axf[i+l].set_yticks((1,2), (f'{["Healthy","Alive"][k==len(labels)-1]}',f'{["Diseased","Deceased"][k==len(labels)-1]}'))
        y = auc(x[len(wk[0]):][a[len(wk[0]):]], x[:len(wk[0])][a[:len(wk[0])]])
        
        foo =["dis'd","dec'd"]
        axf[i].text(0,.9, s= f'{len(x[len(wk[0]):][a[len(wk[0]):]])} {["healthy","alive"][k==len(labels)-1]}\n{len(x[:len(wk[0])][a[:len(wk[0])]])} {foo[k==len(labels)-1]}', transform=axf[i].transAxes, va='top')
        axf[i+l].text(0.5, .8, s = f"AUC={y:.2}",  transform=axf[i+l].transAxes,  va='center', ha='center')
        axf[i+l].set_xlabel(f'Predicted rate [1/yr]')
        axf[i+l].set_ylim((0.5,3.5))
        axf[i].text(0.5, 1, s = f'{aa}-{aa+age_step}yr', transform=axf[i].transAxes, va='bottom', ha='center', weight='bold')

        
        xa = x[a]
        ya = np.concatenate([np.ones(len(wk[0])), np.zeros(x.shape[0] - len(wk[0]))])[a] #* (zk - z)[a] 
        
        if len(xa) == 0:
            continue
        
        if calibration == 'bins':
            if binning == 'deciles':
                bins = np.quantile(xa, np.arange(0,1.05,0.05))
            else:
                bins = bins
            pred = np.array([xa[np.logical_and(xa > bins[b-1], xa <= bins[b])].mean() for b in range(1,len(bins))])
            obs = np.array([ya[np.logical_and(xa > bins[b-1], xa <= bins[b])].mean()  for b in range(1,len(bins))])
            ci = np.array([scipy.stats.beta(0.1 + ya[np.logical_and(xa > bins[b-1], xa <= bins[b])].sum(), 0.1 + (1-ya[np.logical_and(xa > bins[b-1], xa <= bins[b])]).sum()).ppf([0.025,0.975]) for b in range(1,len(bins))])
            axf[i].scatter(pred, obs + 1e-5, marker='.', c='k')
            for j,pr in enumerate(pred):
                if not np.isnan(obs[j]):
                    axf[i].plot( np.repeat(pr,2),ci[j], c='k', lw=.5, ls=":")
            wgt = np.array([[ya[np.logical_and(xa > bins[b-1], xa <= bins[b])].sum(),np.logical_and(xa > bins[b-1], xa <= bins[b]).sum()]  for b in range(1,len(bins))])
            out.append([pred, obs, ci, wgt])
        else:
            o = np.argsort(xa)
            axf[i].plot(xa[o], ya[o]/(ya.sum() - np.cumsum(ya[o]))/age_step, ds='steps')
            out.append(np.nan)
        
        axf[i].set_box_aspect(1)
        axf[i].scatter(xa.mean(), ya.mean(), c='r', ec='w')
        axf[i].set_yscale('log')
        axf[i].set_xscale('log')
        axf[i].set_ylim((1e-5, 1))
        axf[i].set_xlim((1e-5, 1))
        axf[i].plot([0, 1], [0, 1], transform=axf[i].transAxes, lw=.5, c='k', ls="--")

    return out    

In [ ]:
out = []
plt.rcParams.update({'figure.max_open_warning': 0})
for j,k in enumerate(diseases_of_interest):
    for i in [1,0]:
        ii = (d100k[0] == 2+i).sum(1)>0
        out.append(plot_calibration(j,k,(d100k[0][ii].cpu(),d100k[1][ii].cpu(),d100k[2][ii].cpu(),d100k[3][ii].cpu()),p100k[ii.cpu()], age_groups=np.arange(40,80,5), offset=0.1, lifestyle_ix=None))
        plt.gcf().axes[0].set_title(f"{labels.loc[k,0]}, {['females','males'][i]}\n",fontsize=10, weight='bold', loc='left', size=12)
        plt.show()

In [ ]:
import matplotlib.colors
fig, ax = plt.subplots(2,5,figsize=(18,6), sharex=True, sharey=True)
ax=ax.ravel()
for i,x in enumerate(out):
    j = i // 2 # Females and males
    for k,xx in enumerate(x):
        ax[j].plot(xx[0], xx[1]**2/xx[1], label=f"{40+5*k}-{40+5*(k+1)}yrs" , 
                   c=matplotlib.colors.LinearSegmentedColormap.from_list('cmap',list(zip([0,.5,1],[['white','#8520F1','#7A00BF'],['white','#0FB8A1','#00574A']][1-i%2])))(0.15+k/9*.55)) 
    ax[j].set_yscale('log')
    ax[j].set_xscale('log')
    ax[j].set_xlim((5e-5, 0.5))
    ax[j].set_ylim((5e-5, 0.5))
    ax[j].plot([0, 1], [0, 1], transform=ax[j].transAxes, lw=.5, c='k', ls="--")
    ax[j].set_title("\n".join(textwrap.wrap(delphi_labels['name'].iloc[diseases_of_interest[j]],30)),verticalalignment='top', size=10, weight='bold')
    if j >4:
        ax[j].set_xlabel('Model rate [1/yr]')
    if i in [0,5]:
        ax[i].set_ylabel('Observed rate [1/yr]')
    if i==19:
        ax[j].legend(loc='lower left', ncol=2, bbox_to_anchor=(1.05, 0))

    #plt.show()

plt.gcf().tight_layout(h_pad=0.5)
plt.show()

## Loss and predictability of tokens

### Observed loss

In [ ]:
p_obs = []
device = 'cuda'
model.to(device)
batch_size=512
with torch.no_grad():
    for dd in tqdm(zip(*map(lambda x: torch.split(x, batch_size), d100k)), total=d100k[0].shape[0]//batch_size+1):
        p0 = model(*[x.to(device) for x in dd])[0]
        p_obs.append(p0.cpu().gather(2,dd[2].type(torch.int64).unsqueeze(-1).cpu()).squeeze().cpu().detach().numpy())
p_obs = np.vstack(p_obs)

In [ ]:
r_obs =1-np.exp(-np.exp(p_obs)*365.25)

In [ ]:
p_obs_k = np.array([np.nanmedian(p_obs[g][d100k[2].cpu().detach().numpy()[g]==k]) 
                    for k in range(len(labels))  for g in [(d100k[0].cpu().detach().numpy()==gg).sum(1) > 0 
                               for gg in [2,3]]]).reshape(-1,2)

In [ ]:
p_obs_labels = pd.DataFrame(p_obs_k, index=labels[0].to_list(), columns=['Female','Male'])

### Age and sex baseline

In [ ]:
incidence_k_g = []
for k in range(len(labels)):
    h_f,x = np.histogram(females[females[:,2]==k-1,1]/365.25, np.arange(100))
    h_m,x = np.histogram(males[males[:,2]==k-1,1]/365.25, np.arange(100))
    incidence_k_g.append([h_f/n_females,h_m/n_males])

In [ ]:
incidence_k_g = np.array(incidence_k_g)

In [ ]:
p_exp = []
for i in range(d100k[0].shape[0]):
    if 2 in d100k[0][i]:
        j = 0
    elif 3 in d100k[0][i]:
        j = 1
    else:
        p_exp.append(np.repeat(np.nan, d100k[0].shape[1]))
        continue
    ix = d100k[2][i].cpu().detach().numpy()
    iy = (d100k[3][i].cpu().detach().numpy()/365.25).astype('int')  
    p_exp.append(incidence_k_g[ix,j,iy])

In [ ]:
p_exp = np.array(p_exp)

In [ ]:
lodds_obs_exp = np.log(r_obs) - np.log(p_exp + 1e-6)

In [ ]:
lodds_obs_exp_k = np.array([np.nanmean(lodds_obs_exp[g][d100k[2].cpu().detach().numpy()[g]==k]) 
                            for k in range(len(labels))  for g in [(d100k[0].cpu().detach().numpy()==gg).sum(1) > 0                            for gg in [2,3]]]).reshape(-1,2)

In [ ]:
lodds_obs_exp_labels = pd.DataFrame(lodds_obs_exp_k, index=labels[0].to_list(), columns=['Female','Male'])

In [ ]:
n_k = np.array([np.histogram(females[:,-1]+1, range(len(labels)+1))[0],
                np.histogram(males[:,-1]+1, range(len(labels)+1))[0]])

In [ ]:
df = np.exp(lodds_obs_exp_labels).merge(pd.DataFrame(n_k.T, index=labels[0]), right_index=True, left_index=True)
df.columns = ['RR Female','RR Male', 'n Female','n Male']

In [ ]:
df['n_total'] = df['n Female'] + df['n Male'] 
df = df.reset_index()

In [ ]:
df

In [ ]:
plt.figure(figsize=(5,3.5))
for i in range(2):
    plt.scatter(n_k[i,13:], np.exp(lodds_obs_exp_labels.iloc[13:,i]), marker='.',s=50, ec='white', fc=['#8520F1','#0FB8A1'][i], label=['Female','Male'][i])
    
plt.yscale('log')
plt.xscale('log')
plt.xlabel('Number of tokens')
plt.ylabel('Relative risk')
plt.axhline(1, c='k',ls="--", lw=0.75)
plt.legend()

plt.show()

In [ ]:
chapter = lodds_obs_exp_labels.iloc[4:].index.str.slice(0,1).to_list()
chapter[-1]='X'

In [ ]:
chapters = """I Certain infectious and parasitic diseases
II Neoplasms
III Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism
IV Endocrine, nutritional and metabolic diseases
V Mental and behavioural disorders
VI Diseases of the nervous system
VII Diseases of the eye and adnexa
VIII Diseases of the ear and mastoid process
IX Diseases of the circulatory system
X Diseases of the respiratory system
XI Diseases of the digestive system
XII Diseases of the skin and subcutaneous tissue
XIII Diseases of the musculoskeletal system and connective tissue
XIV Diseases of the genitourinary system
XV Pregnancy, childbirth and the puerperium
XVI Certain conditions originating in the perinatal period
XVII Congenital malformations, deformations and chromosomal abnormalities
XVIII Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified
XIX Injury, poisoning and certain other consequences of external causes
XX External causes of morbidity and mortality
XXI Factors influencing health status and contact with health services
XXII Codes for special purposes""".split("\n")

In [ ]:
chapter_order = ['Technical', 'Sex', 'Smoking, Alcohol and BMI',
       'I. Infectious Diseases', 'II. Neoplasms', 'III. Blood & Immune Disorders',
       'IV. Metabolic Diseases', 'V. Mental Disorders',
       'VI. Nervous System Diseases', 'VII. Eye Diseases',
       'VIII. Ear Diseases', 'IX. Circulatory Diseases',
       'X. Respiratory Diseases', 'XI. Digestive Diseases',
       'XII. Skin Diseases', 'XIII. Musculoskeletal Diseases',
       'XIV. Genitourinary Diseases', 'XV. Pregnancy & Childbirth',
       'XVI. Perinatal Conditions', 'XVII. Congenital Abnormalities', 'Death']

In [ ]:
foo = lodds_obs_exp_labels.copy()
foo.iloc[n_k.T < 100] = np.nan
foo.index.name='ICD10'
foo['Chapter'] = delphi_labels['ICD-10 Chapter (short)'].values
foo = foo.iloc[13:-1]
foo = foo.reset_index().set_index('Chapter').sort_index(key=lambda x: x.map(lambda y: chapter_order.index(y)))
foo = foo.reset_index().set_index(['Chapter', 'ICD10'])

In [ ]:
whisker_colour = (0.4, 0.4, 0.4)
bplots = np.exp(foo).groupby('Chapter').boxplot(figsize=(8,3.5), subplots=False, 
                                                return_type='both', 
                                                showfliers=False,
                                                boxprops={'linewidth': 1.25},
                                                medianprops={'linewidth': 1, 'color': whisker_colour},
                                                whiskerprops={'linewidth': 1, 'color': whisker_colour},
                                                capprops={'linewidth': 1, 'color': (0,0,0,0)},
                                                patch_artist=True, whis=[2.5, 97.5])

for item in ['boxes', 'whiskers']:
    for i, box in enumerate(bplots[1][item]):
        try:
            box.set_edgecolor(['#8520F1','#0FB8A1'][i%2])
            box.set_facecolor('white')
        except:
            box.set_color(['#8520F1','#0FB8A1'][i%4 >= 2])
plt.ylim(0.8, 100)
plt.yscale('log')
xticks = np.array(plt.gca().get_xticks()[::2]) + 0.5
xtickslabels = plt.gca().get_xticklabels()[::2]
xtickslabels = [i._text.replace('(', '').replace(', Female)', '') for i in xtickslabels]

plt.gca().set_xticks(xticks)
plt.gca().set_xticklabels(xtickslabels)

plt.xticks(rotation=90)
plt.ylabel('Relative risk')
plt.axhline(1, c='k', ls='--')
plt.gca().tick_params(length=1.75, width=0.5, labelsize=10, grid_alpha=0.0, grid_linewidth=0.45, axis='x')
plt.scatter([], [], c='#8520F1', label='Female')
plt.scatter([], [], c='#0FB8A1', label='Male')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

for i in range(1, 34, 4):
    plt.fill_between([i - 0.5, i + 1.5], 0.8, 150, color=(0.945, 0.945, 0.945))

plt.show()

## Interpretability

### Attention maps

In [ ]:
d = get_batch([0], val, val_p2i,  select='smart_random', block_size=model.config.block_size, device=device)

risk = np.exp(model(d[0],d[1])[0].cpu().detach().numpy().squeeze())
att = model(d[0],d[1])[2].cpu().detach().numpy().squeeze()
fig, ax = plt.subplots(*att.shape[:2], figsize=(12,12), sharex=True, sharey=True)
for i in range(att.shape[0]):
    for j in range(att.shape[1]):
        ax[i,j].imshow(att[i,j], vmax=0.35)
        if i==0:
            ax[i,j].set_title(f"Head {j}")
        if j==0:
            ax[i,j].set_ylabel(f"Layer {i}")
plt.tight_layout()

In [ ]:
d = get_batch(range(dataset_subset_size), val, val_p2i,  
              select='smart_random', block_size=48, 
              device=device, padding='random')
w = np.where(torch.isin(d[2].cpu(), torch.tensor(diseases_of_interest)).sum(axis=1))
w = (w[0][:3],)
att = model(*list(map(lambda x: x[w[0],:], d)))[2].cpu().detach().numpy().squeeze()
att.shape

In [ ]:
import textwrap

d = [d_.cpu() for d_ in d]

for i in range(len(w[0])):
    print(i)
    j = (d[0][w[0][i]]==0).sum()
    plt.figure(figsize=(3 * (d[3][w[0][i],-1]-d[1][w[0][i],0])/365.25/70,9 * (48-j)/48))
    x = torch.concatenate([d[1][w[0][i]], d[3][w[0][i],[-1]]])/365.25
    plt.pcolormesh( x[j:],np.arange(j,49,1), att[0,i,:,j:,j:].max((0)).T, cmap='Blues')
    _ = plt.yticks(np.arange(j,48)+.5, [f"{textwrap.shorten(delphi_labels.loc[i,'name'],50)}" if i > 1 else "" for i,t in zip(d[0][w[0][i],j:].detach().numpy().squeeze(),d[1][w[0][i],j:].detach().numpy().squeeze()/365.25)])
    plt.gca().invert_yaxis()
    plt.xlabel('Age')
    plt.show()

## Embeddings

In [ ]:
import umap
import matplotlib as mpl

In [ ]:
wte = model.transformer.wte.weight.cpu().detach().numpy()
seed = 1413
t  = umap.UMAP(random_state=seed, n_neighbors=30, min_dist=0.05, metric='cosine').fit(wte)

u0 = t.transform(model.transformer.wte.weight.cpu().detach().numpy())#+ wae[70].detach().numpy())
u = u0 - np.median(u0, axis=0)
u = - u

In [ ]:
def remove_ticks(ax):
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    
    for tick in ax.xaxis.get_major_ticks():
        tick.tick1line.set_visible(False)
        tick.tick2line.set_visible(False)
    
    for tick in ax.yaxis.get_major_ticks():
        tick.tick1line.set_visible(False)
        tick.tick2line.set_visible(False)

In [ ]:
labels_all = pd.read_csv('delphi_labels_chapters_colours_icd.csv')
labels_all['UMAP1'] =  u[:,0]
labels_all['UMAP2'] =  u[:,1]
labels_all = labels_all[labels_all['count'] > 20].reset_index(drop=True).reset_index()
labels_non_technical = labels_all[~labels_all['ICD-10 Chapter'].isin(['Technical', 'Sex', 'Smoking, Alcohol and BMI'])]
labels_non_technical = labels_non_technical[(labels_non_technical['UMAP1'].abs() < 5) & (labels_non_technical['UMAP2'].abs() < 5)]
short_names = labels_all['ICD-10 Chapter (short)'].unique()
short_names_present = [i for i in short_names if i in labels_non_technical['ICD-10 Chapter (short)'].unique()]
color_mapping_short = {k: v for k, v in labels_all[['ICD-10 Chapter (short)', 'color']].values}

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 8))

sns.scatterplot(x='UMAP1', y='UMAP2', data=labels_non_technical, hue='ICD-10 Chapter (short)', 
                palette=color_mapping_short, 
                hue_order=short_names_present, size='count', sizes=(20, 200), 
                alpha=0.9, ax=ax, linewidth=0.15)

ax.legend_.set_bbox_to_anchor((1.1, 0.85))
ax.grid(None)
remove_ticks(ax)
ax.set_aspect('equal')
plt.title('UMAP of learned disease embeddings');

In [ ]:
def embd(model, idx, age, targets, targets_age):
    with torch.no_grad():
        device = idx.device
        b, t = idx.size()
        assert t <= model.config.block_size, f"Cannot forward sequence of length {t}, block size is only {model.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        tok_emb = model.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        age_emb = model.transformer.wae(age.unsqueeze(-1)) # age embeddings of shape (b, t, n_embd)
        x = model.transformer.drop(tok_emb + age_emb)
        
        attn_mask = (idx>0).view(idx.size(0), 1, 1, idx.size(1)) * (idx>0).view(idx.size(0),1,idx.size(1),1)  # Do not attend to padded positions
        attn_mask *= torch.tril(torch.ones(idx.size(1),idx.size(1), device=device))[None,None,:,:] > 0 #self.transformer.h[0].attn.bias[:,:,:idx.size(1),:idx.size(1)] > 0
        if targets is not None and model.config.mask_ties:
            attn_mask *= ((age.view(idx.size(0),1,1,idx.size(1)) != targets_age.view(idx.size(0),1,idx.size(1),1))) # Mask co-occuring tokens
            attn_mask += (attn_mask.sum(-1, keepdim=True)==0) * torch.diag(torch.ones(idx.size(1), device=device)) > 0
        attn_mask = attn_mask + (idx==0).view(idx.size(0), 1, 1, idx.size(1)) * torch.diag(torch.ones(idx.size(1), device=device)) > 0 # Except for padding
        attn_mask *= torch.tril(torch.ones(idx.size(1),idx.size(1), device=device))[None,None,:,:] > 0 #self.transformer.h[0].attn.bias[:,:,:idx.size(1),:idx.size(1)] > 0

        
        for block in model.transformer.h:
            x, a = block(x, attn_mask)
        x = model.transformer.ln_f(x)

        return (tok_emb+age_emb).cpu().detach().numpy(), x[:,:,:].cpu().detach().numpy()

In [ ]:
d_10k = get_batch(range(dataset_subset_size), val, val_p2i,  
              select='smart_random', block_size=64, 
              device=device, padding='random')

In [ ]:
diseases_of_interest = [46, 95, 1168, 1188, 374, 214, 305, 505, 584]

In [ ]:
trajectory_of_interest = [torch.where((d_10k[2] == k).any(-1))[0][9] for k in diseases_of_interest]
d = get_batch(trajectory_of_interest, val, val_p2i,  
              select='smart_random', block_size=48, 
              device=device, padding='random')
e0, e = embd(model, *[d_ for d_ in d]) # calculate initial and final embeddings

In [ ]:
import seaborn as sns

fig, axs = plt.subplots(3, 3, figsize=(14, 14))

axs = axs.flatten()

for ax, traj, k in zip(axs, e, diseases_of_interest):
    sns.scatterplot(x='UMAP1', y='UMAP2', data=labels_non_technical, hue='ICD-10 Chapter (short)', 
                    palette=color_mapping_short, legend=None, 
                    hue_order=short_names_present, size='count', sizes=(12, 80), 
                    alpha=0.9, ax=ax, linewidth=0.15, rasterized=True)

    ax.grid(None)
    ax.axis('off')
    ax.set_aspect('equal')

    e_projected = t.transform(traj)
    e_projected = e_projected - np.median(u0, axis=0)
    e_projected = - e_projected
    ax.plot(*e_projected.T, color='k', linewidth=0.8)
    ax.scatter(e_projected[0, 0], e_projected[0, 1], color='blue', s=64, marker=9, zorder=10, label='Trajectory start')
    ax.scatter(e_projected[-1, 0], e_projected[-1, 1], color='red', s=64, marker='*', zorder=10, label='Trajectory end')
    ax.text(u[:,0][-1], u[:,1][-1], 'Death', va='center', ha='left')
    ax.text(u[:,0][k], u[:,1][k], delphi_labels.loc[k, 'name'])
ax.legend();